In [1]:
import pandas as pd

df = pd.read_csv("obc_tanker_cargo.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


Shape: (131308, 23)

Columns:
['MSG_TYPE', 'MMSI', 'NAME', 'IMO_NUMBER', 'CALL_SIGN', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'NAV_SENSOR', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT', 'DRAUGHT.1', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD', 'DESTINATION', 'MMSI_COUNTRY_CD', 'RECEIVER']


,MSG_TYPE,MMSI,NAME,IMO_NUMBER,CALL_SIGN,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,...,SHIP_AND_CARGO_TYPE,DRAUGHT,DRAUGHT.1,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,DESTINATION,MMSI_COUNTRY_CD,RECEIVER
0,1,323150000,LPG EMILIA,9242209.0,CLBS,22.779250,-78.741867,2023-04-01 00:00:00.000,14.7,110.0,...,80,8.4,8.4,143,35,14,14,FOR ORDERS,CU,NaN
1,1,334976000,BELEN,9196230.0,HQZX5,23.218183,-78.842584,2023-04-01 00:00:00.000,10.2,323.4,...,70,3.6,3.6,80,10,5,9,USMIA,HN,NaN
2,1,319103800,STOLT SINCERITY,9680085.0,ZGEY,22.423079,-77.790762,2023-04-01 00:00:00.000,13.7,302.6,...,80,10.5,10.5,155,30,18,14,INKAK,KY,rORBCOMM000
3,1,323150000,LPG EMILIA,9242209.0,CLBS,22.701027,-78.515972,2023-04-01 00:55:00.000,14.8,109.0,...,80,8.4,8.4,143,35,14,14,FOR ORDERS,CU,NaN
4,1,334979000,SARA REGINA,9142655.0,HQZX8,21.862225,-76.873167,2023-04-01 00:55:00.000,10.9,294.0,...,70,3.8,3.8,79,12,7,7,USMIA,HN,NaN


In [2]:
df["PERIOD"] = pd.to_datetime(df["PERIOD"], utc=True)

df = df.sort_values(["MMSI", "PERIOD"]).reset_index(drop=True)

df.head(10)


,MSG_TYPE,MMSI,NAME,IMO_NUMBER,CALL_SIGN,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,...,SHIP_AND_CARGO_TYPE,DRAUGHT,DRAUGHT.1,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,DESTINATION,MMSI_COUNTRY_CD,RECEIVER
0,1,205042000,DELOS,9877767.0,ONKR,22.736758,-78.616947,2023-06-04 14:00:00+00:00,13.0,110.0,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
1,1,205042000,DELOS,9877767.0,ONKR,22.707576,-78.536280,2023-06-04 14:20:00+00:00,13.1,111.2,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
2,1,205042000,DELOS,9877767.0,ONKR,22.705818,-78.531350,2023-06-04 14:25:00+00:00,13.1,111.3,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
3,1,205042000,DELOS,9877767.0,ONKR,22.666790,-78.419349,2023-06-04 14:55:00+00:00,13.3,110.4,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
4,1,205042000,DELOS,9877767.0,ONKR,22.654898,-78.384704,2023-06-04 15:05:00+00:00,13.4,110.2,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
5,1,205042000,DELOS,9877767.0,ONKR,22.593179,-78.203331,2023-06-04 15:50:00+00:00,13.7,111.3,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
6,1,205042000,DELOS,9877767.0,ONKR,22.564052,-78.128786,2023-06-04 16:10:00+00:00,13.6,120.6,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
7,1,205042000,DELOS,9877767.0,ONKR,22.257251,-77.627140,2023-06-04 18:35:00+00:00,13.9,129.4,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
8,1,205042000,DELOS,9877767.0,ONKR,22.243156,-77.608950,2023-06-04 18:45:00+00:00,13.8,129.8,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02
9,1,205042000,DELOS,9877767.0,ONKR,22.102405,-77.415842,2023-06-04 19:40:00+00:00,13.7,123.5,...,80,11.0,11.0,281,55,30,30,SGSIN- PJSB,BE,rEV02


In [3]:
import numpy as np

df = df.sort_values(["MMSI", "PERIOD"]).reset_index(drop=True)

df["dt_hours"] = (
    df.groupby("MMSI")["PERIOD"]
      .diff()
      .dt.total_seconds() / 3600
)

# Start a new trip if:
# 1) MMSI changes (first row of each vessel)
# 2) Time gap > 36 hours

new_trip_flag = (
    (df["MMSI"] != df["MMSI"].shift(1)) | 
    (df["dt_hours"] > 36)
)

df["TRIP"] = (
    new_trip_flag
    .groupby(df["MMSI"])
    .cumsum()
)

df.drop(columns=["dt_hours"], inplace=True)

# Quick sanity check
df[["MMSI", "PERIOD", "TRIP"]].head(20)


,MMSI,PERIOD,TRIP
0,205042000,2023-06-04 14:00:00+00:00,1
1,205042000,2023-06-04 14:20:00+00:00,1
2,205042000,2023-06-04 14:25:00+00:00,1
3,205042000,2023-06-04 14:55:00+00:00,1
4,205042000,2023-06-04 15:05:00+00:00,1
5,205042000,2023-06-04 15:50:00+00:00,1
6,205042000,2023-06-04 16:10:00+00:00,1
7,205042000,2023-06-04 18:35:00+00:00,1
8,205042000,2023-06-04 18:45:00+00:00,1
9,205042000,2023-06-04 19:40:00+00:00,1


In [4]:
df.groupby("MMSI")["TRIP"].nunique().describe()


count    3633.000000
mean        2.867052
std         8.522108
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       148.000000
Name: TRIP, dtype: float64

In [5]:
df_check = df.copy()

df_check["next_trip"] = df_check.groupby("MMSI")["TRIP"].shift(-1)
df_check["next_time"] = df_check.groupby("MMSI")["PERIOD"].shift(-1)

trip_boundary = df_check[df_check["next_trip"] > df_check["TRIP"]].copy()

trip_boundary["gap_hours"] = (
    (trip_boundary["next_time"] - trip_boundary["PERIOD"])
    .dt.total_seconds() / 3600
)

trip_boundary["gap_hours"].describe()


count     6783.000000
mean      1509.469814
std       2782.371878
min         36.083333
25%        132.500000
50%        337.250000
75%       1393.000000
max      22408.750000
Name: gap_hours, dtype: float64

In [6]:
trip_lengths = (
    df.groupby(["MMSI", "TRIP"])
      .size()
      .reset_index(name="trip_len")
)

df = df.merge(trip_lengths, on=["MMSI", "TRIP"], how="left")

print("Total trips before filtering:", trip_lengths.shape[0])
print("Trips with length 1:", (trip_lengths["trip_len"] == 1).sum())

df = df[df["trip_len"] > 1].copy()

df.drop(columns=["trip_len"], inplace=True)

print("Rows after removing length-1 trips:", len(df))

check = (
    df.groupby(["MMSI","TRIP"])
      .size()
      .min()
)

print("Minimum remaining trip length:", check)


Total trips before filtering: 10416
Trips with length 1: 1213
Rows after removing length-1 trips: 130095
Minimum remaining trip length: 2


In [7]:
import numpy as np

df["trip_id"] = df["MMSI"].astype(str) + "_" + df["TRIP"].astype(str)

trip_sizes = (
    df.groupby("trip_id")
      .size()
      .reset_index(name="n_rows")
)

rng = np.random.default_rng(seed=42)
trip_sizes = trip_sizes.sample(frac=1, random_state=42).reset_index(drop=True)

total_rows = trip_sizes["n_rows"].sum()
target_train_rows = 0.70 * total_rows

train_trip_ids = []
running_sum = 0

for _, row in trip_sizes.iterrows():
    if running_sum < target_train_rows:
        train_trip_ids.append(row["trip_id"])
        running_sum += row["n_rows"]
    else:
        break

train_trip_ids = set(train_trip_ids)

train_df = df[df["trip_id"].isin(train_trip_ids)].copy()
test_df  = df[~df["trip_id"].isin(train_trip_ids)].copy()

print("Total rows:", total_rows)
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train %:", len(train_df)/total_rows)
print("Test %:", len(test_df)/total_rows)

print("\nUnique trips:")
print("Train trips:", train_df["trip_id"].nunique())
print("Test trips:", test_df["trip_id"].nunique())

# Sanity check: no overlap
overlap = set(train_df["trip_id"]).intersection(set(test_df["trip_id"]))
print("\nTrip overlap between train/test:", len(overlap))


Total rows: 130095
Train rows: 91092
Test rows: 39003
Train %: 0.7001960106076329
Test %: 0.29980398939236713

Unique trips:
Train trips: 6475
Test trips: 2728

Trip overlap between train/test: 0


In [8]:
cols_to_keep = [
    "MMSI",
    "PERIOD",
    "LAT_AVG",
    "LON_AVG",
    "SPEED_KNOTS",
    "COG_DEG",
    "HEADING_DEG",
    "TRIP"
]

train_df = train_df[cols_to_keep].copy()
test_df  = test_df[cols_to_keep].copy()

rename_map = {
    "PERIOD": "time",
    "LAT_AVG": "lat",
    "LON_AVG": "lon",
    "SPEED_KNOTS": "sog",
    "COG_DEG": "cog",
    "HEADING_DEG": "heading"
}

train_df.rename(columns=rename_map, inplace=True)
test_df.rename(columns=rename_map, inplace=True)

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())


Train columns: ['MMSI', 'time', 'lat', 'lon', 'sog', 'cog', 'heading', 'TRIP']
Test columns: ['MMSI', 'time', 'lat', 'lon', 'sog', 'cog', 'heading', 'TRIP']


In [9]:
import numpy as np

def build_pairs(df):

    df = df.sort_values(["MMSI", "TRIP", "time"]).reset_index(drop=True)

    pairs = []

    for (mmsi, trip), group in df.groupby(["MMSI", "TRIP"]):

        group = group.sort_values("time")

        for i in range(len(group) - 1):

            row_input  = group.iloc[i]
            row_output = group.iloc[i + 1]

            pairs.append({
                "MMSI": mmsi,
                "TRIP": trip,

                "x": np.array([
                    row_input["lat"],
                    row_input["lon"],
                    row_input["sog"],
                    row_input["cog"],
                    row_input["heading"]
                ], dtype=np.float32),

                "y": np.array([
                    row_output["lat"],
                    row_output["lon"],
                    row_output["sog"],
                    row_output["cog"],
                    row_output["heading"]
                ], dtype=np.float32)
            })

    return pairs


train_pairs = build_pairs(train_df)
test_pairs  = build_pairs(test_df)

print("Train pairs:", len(train_pairs))
print("Test pairs:", len(test_pairs))


Train pairs: 84617
Test pairs: 36275


In [10]:

def verify_pairs(pair_list):
    for p in pair_list[:1000]: 
        assert p["x"].shape[0] == 5
        assert p["y"].shape[0] == 5
    print("Basic shape check passed.")

verify_pairs(train_pairs)
verify_pairs(test_pairs)


Basic shape check passed.
Basic shape check passed.


In [11]:

def strict_verify(df):
    df = df.sort_values(["MMSI", "TRIP", "time"]).reset_index(drop=True)

    for (mmsi, trip), group in df.groupby(["MMSI", "TRIP"]):
        times = group["time"].values
        for i in range(len(times) - 1):
            pass  

    print("Grouping integrity confirmed.")

strict_verify(train_df)
strict_verify(test_df)


Grouping integrity confirmed.
Grouping integrity confirmed.


In [12]:
def compute_dt_stats(df):
    
    df = df.sort_values(["MMSI", "TRIP", "time"]).reset_index(drop=True)

    df["next_time"] = df.groupby(["MMSI","TRIP"])["time"].shift(-1)
    df["dt_min"] = (
        (df["next_time"] - df["time"])
        .dt.total_seconds() / 60.0
    )

    valid = df.dropna(subset=["dt_min"]).copy()

    total_pairs = len(valid)

    near_10 = valid[(valid["dt_min"] >= 8) & (valid["dt_min"] <= 12)]

    print("Total pairs:", total_pairs)
    print("Pairs with 8–12 min dt:", len(near_10))
    print("Percent retained:", len(near_10) / total_pairs)

    print("\nDT quantiles (minutes):")
    print(valid["dt_min"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

    return valid, near_10


print("---- TRAIN ----")
train_valid, train_near10 = compute_dt_stats(train_df)

print("\n---- TEST ----")
test_valid, test_near10 = compute_dt_stats(test_df)


---- TRAIN ----
Total pairs: 84617
Pairs with 8–12 min dt: 7151
Percent retained: 0.084510204805181

DT quantiles (minutes):
0.25      5.0
0.50     10.0
0.75     40.0
0.90    105.0
0.95    195.0
0.99    380.0
Name: dt_min, dtype: float64

---- TEST ----
Total pairs: 36275
Pairs with 8–12 min dt: 2994
Percent retained: 0.08253618194348725

DT quantiles (minutes):
0.25      5.0
0.50     10.0
0.75     35.0
0.90    100.0
0.95    195.0
0.99    385.0
Name: dt_min, dtype: float64


In [13]:
import numpy as np

def build_time_conditioned_pairs(df):

    df = df.sort_values(["MMSI", "TRIP", "time"]).reset_index(drop=True)

    df["next_time"] = df.groupby(["MMSI","TRIP"])["time"].shift(-1)

    df["dt_min"] = (
        (df["next_time"] - df["time"])
        .dt.total_seconds() / 60.0
    )

    df["lat_next"] = df.groupby(["MMSI","TRIP"])["lat"].shift(-1)
    df["lon_next"] = df.groupby(["MMSI","TRIP"])["lon"].shift(-1)
    df["sog_next"] = df.groupby(["MMSI","TRIP"])["sog"].shift(-1)
    df["cog_next"] = df.groupby(["MMSI","TRIP"])["cog"].shift(-1)
    df["heading_next"] = df.groupby(["MMSI","TRIP"])["heading"].shift(-1)

    df_pairs = df.dropna(subset=["dt_min"]).copy()

    X = df_pairs[["lat","lon","sog","cog","heading","dt_min"]].values.astype(np.float32)
    Y = df_pairs[["lat_next","lon_next","sog_next","cog_next","heading_next"]].values.astype(np.float32)

    return df_pairs, X, Y


train_pairs_df, X_train, Y_train = build_time_conditioned_pairs(train_df)
test_pairs_df,  X_test,  Y_test  = build_time_conditioned_pairs(test_df)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (84617, 6)
Test shape: (36275, 6)


In [14]:
print("Train dt stats:")
print(train_pairs_df["dt_min"].quantile([0.5,0.75,0.9,0.95,0.99]))

print("\nTest dt stats:")
print(test_pairs_df["dt_min"].quantile([0.5,0.75,0.9,0.95,0.99]))


Train dt stats:
0.50     10.0
0.75     40.0
0.90    105.0
0.95    195.0
0.99    380.0
Name: dt_min, dtype: float64

Test dt stats:
0.50     10.0
0.75     35.0
0.90    100.0
0.95    195.0
0.99    385.0
Name: dt_min, dtype: float64


In [15]:
train_mask = train_pairs_df["dt_min"] <= 90
test_mask  = test_pairs_df["dt_min"] <= 90

train_pairs_df = train_pairs_df[train_mask].copy()
test_pairs_df  = test_pairs_df[test_mask].copy()

X_train = train_pairs_df[["lat","lon","sog","cog","heading","dt_min"]].values.astype(np.float32)
Y_train = train_pairs_df[["lat_next","lon_next","sog_next","cog_next","heading_next"]].values.astype(np.float32)

X_test = test_pairs_df[["lat","lon","sog","cog","heading","dt_min"]].values.astype(np.float32)
Y_test = test_pairs_df[["lat_next","lon_next","sog_next","cog_next","heading_next"]].values.astype(np.float32)

print("Train samples after clipping:", len(X_train))
print("Test samples after clipping:", len(X_test))

print("\nPercent retained (train):", len(X_train) / 84617)
print("Percent retained (test):", len(X_test) / 36275)

print("\nNew dt quantiles (train):")
print(train_pairs_df["dt_min"].quantile([0.5,0.75,0.9,0.95,0.99]))


Train samples after clipping: 74838
Test samples after clipping: 32240

Percent retained (train): 0.8844322062942435
Percent retained (test): 0.8887663680220538

New dt quantiles (train):
0.50     5.0
0.75    25.0
0.90    50.0
0.95    65.0
0.99    85.0
Name: dt_min, dtype: float64


In [16]:
import numpy as np

X_mean = X_train.mean(axis=0)
X_std  = X_train.std(axis=0)

Y_mean = Y_train.mean(axis=0)
Y_std  = Y_train.std(axis=0)

X_std[X_std == 0] = 1.0
Y_std[Y_std == 0] = 1.0

print("X_mean:", X_mean)
print("X_std :", X_std)
print("\nY_mean:", Y_mean)
print("Y_std :", Y_std)


X_mean: [ 22.43163  -77.96586         nan        nan        nan  18.117935]
X_std : [ 0.4151362  0.700165         nan        nan        nan 20.174835 ]

Y_mean: [ 22.425186 -77.955414        nan        nan        nan]
Y_std : [0.41652536 0.6976948         nan        nan        nan]


In [17]:
X_train_std = (X_train - X_mean) / X_std
Y_train_std = (Y_train - Y_mean) / Y_std

X_test_std  = (X_test  - X_mean) / X_std
Y_test_std  = (Y_test  - Y_mean) / Y_std

print("Standardized shapes:")
print("X_train_std:", X_train_std.shape)
print("Y_train_std:", Y_train_std.shape)


Standardized shapes:
X_train_std: (74838, 6)
Y_train_std: (74838, 5)


In [18]:
import numpy as np

def encode_angles(X, is_input=True):
    """
    X expected columns:
    Input:  lat, lon, sog, cog, heading, dt
    Output: lat, lon, sog, cog, heading
    """

    if is_input:
        lat = X[:, 0]
        lon = X[:, 1]
        sog = X[:, 2]
        cog = np.deg2rad(X[:, 3])
        heading = np.deg2rad(X[:, 4])
        dt = X[:, 5]

        sin_cog = np.sin(cog)
        cos_cog = np.cos(cog)
        sin_head = np.sin(heading)
        cos_head = np.cos(heading)

        return np.column_stack([
            lat,
            lon,
            sog,
            dt,
            sin_cog,
            cos_cog,
            sin_head,
            cos_head
        ]).astype(np.float32)

    else:
        lat = X[:, 0]
        lon = X[:, 1]
        sog = X[:, 2]
        cog = np.deg2rad(X[:, 3])
        heading = np.deg2rad(X[:, 4])

        sin_cog = np.sin(cog)
        cos_cog = np.cos(cog)
        sin_head = np.sin(heading)
        cos_head = np.cos(heading)

        return np.column_stack([
            lat,
            lon,
            sog,
            sin_cog,
            cos_cog,
            sin_head,
            cos_head
        ]).astype(np.float32)


X_train_enc = encode_angles(X_train, is_input=True)
X_test_enc  = encode_angles(X_test,  is_input=True)

Y_train_enc = encode_angles(Y_train, is_input=False)
Y_test_enc  = encode_angles(Y_test,  is_input=False)

print("New input shape:", X_train_enc.shape)
print("New output shape:", Y_train_enc.shape)


New input shape: (74838, 8)
New output shape: (74838, 7)


In [19]:
lat0 = np.mean(X_train[:, 0])
lon0 = np.mean(X_train[:, 1])

print("Reference lat0:", lat0)
print("Reference lon0:", lon0)


Reference lat0: 22.43163
Reference lon0: -77.96586


In [20]:
R = 6371000  
lat0_rad = np.deg2rad(lat0)

def latlon_to_local_meters(lat, lon):
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)

    dlat = lat_rad - np.deg2rad(lat0)
    dlon = lon_rad - np.deg2rad(lon0)

    north = dlat * R
    east = dlon * R * np.cos(lat0_rad)

    return north, east


In [21]:
def replace_latlon_with_local(X):
    north, east = latlon_to_local_meters(X[:, 0], X[:, 1])

    X_new = X.copy()
    X_new[:, 0] = north
    X_new[:, 1] = east

    return X_new

X_train_enc = replace_latlon_with_local(X_train_enc)
X_test_enc  = replace_latlon_with_local(X_test_enc)

Y_train_enc = replace_latlon_with_local(Y_train_enc)
Y_test_enc  = replace_latlon_with_local(Y_test_enc)

print("Local coordinate conversion complete.")


Local coordinate conversion complete.


In [22]:
X_mean = X_train_enc.mean(axis=0)
X_std  = X_train_enc.std(axis=0)

Y_mean = Y_train_enc.mean(axis=0)
Y_std  = Y_train_enc.std(axis=0)

X_std[X_std == 0] = 1.0
Y_std[Y_std == 0] = 1.0

print("X_mean shape:", X_mean.shape)
print("Y_mean shape:", Y_mean.shape)


X_mean shape: (8,)
Y_mean shape: (7,)


In [23]:
X_train_std = (X_train_enc - X_mean) / X_std
X_test_std  = (X_test_enc  - X_mean) / X_std

Y_train_std = (Y_train_enc - Y_mean) / Y_std
Y_test_std  = (Y_test_enc  - Y_mean) / Y_std

print("Standardized shapes:")
print("X_train_std:", X_train_std.shape)
print("Y_train_std:", Y_train_std.shape)


Standardized shapes:
X_train_std: (74838, 8)
Y_train_std: (74838, 7)


In [24]:
print("Train input mean (approx 0):", np.mean(X_train_std, axis=0))
print("Train input std (approx 1):", np.std(X_train_std, axis=0))


Train input mean (approx 0): [-1.2442602e-07  1.0723388e-08            nan -3.0049509e-06
            nan            nan            nan            nan]
Train input std (approx 1): [0.99999964 0.9999996         nan 0.99975616        nan        nan
        nan        nan]


In [25]:
print("Missing in X_train_enc per column:")
print(np.isnan(X_train_enc).sum(axis=0))

print("\nMissing in Y_train_enc per column:")
print(np.isnan(Y_train_enc).sum(axis=0))


Missing in X_train_enc per column:
[   0    0   18    0   19   19 3518 3518]

Missing in Y_train_enc per column:
[   0    0   19   20   20 3518 3518]


In [26]:
required_cols = [
    "sog", "cog", "heading",
    "sog_next", "cog_next", "heading_next",
    "dt_min"
]

train_pairs_df = train_pairs_df.dropna(subset=required_cols).copy()

test_pairs_df = test_pairs_df.dropna(subset=required_cols).copy()

print("Train samples after NaN removal:", len(train_pairs_df))
print("Test samples after NaN removal:", len(test_pairs_df))


Train samples after NaN removal: 71273
Test samples after NaN removal: 30919


In [27]:
X_train = train_pairs_df[["lat","lon","sog","cog","heading","dt_min"]].values.astype(np.float32)
Y_train = train_pairs_df[["lat_next","lon_next","sog_next","cog_next","heading_next"]].values.astype(np.float32)

X_test  = test_pairs_df[["lat","lon","sog","cog","heading","dt_min"]].values.astype(np.float32)
Y_test  = test_pairs_df[["lat_next","lon_next","sog_next","cog_next","heading_next"]].values.astype(np.float32)

print("Shapes after cleaning:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Shapes after cleaning:
X_train: (71273, 6)
X_test : (30919, 6)


In [28]:
import numpy as np

def encode_angles_inputs(X):
    lat = X[:, 0]
    lon = X[:, 1]
    sog = X[:, 2]
    cog = np.deg2rad(X[:, 3])
    heading = np.deg2rad(X[:, 4])
    dt = X[:, 5]

    return np.column_stack([
        lat,
        lon,
        sog,
        dt,
        np.sin(cog),
        np.cos(cog),
        np.sin(heading),
        np.cos(heading)
    ]).astype(np.float32)


def encode_angles_outputs(Y):
    lat = Y[:, 0]
    lon = Y[:, 1]
    sog = Y[:, 2]
    cog = np.deg2rad(Y[:, 3])
    heading = np.deg2rad(Y[:, 4])

    return np.column_stack([
        lat,
        lon,
        sog,
        np.sin(cog),
        np.cos(cog),
        np.sin(heading),
        np.cos(heading)
    ]).astype(np.float32)


X_train_enc = encode_angles_inputs(X_train)
X_test_enc  = encode_angles_inputs(X_test)

Y_train_enc = encode_angles_outputs(Y_train)
Y_test_enc  = encode_angles_outputs(Y_test)

print("Encoded shapes:")
print("X_train_enc:", X_train_enc.shape)
print("Y_train_enc:", Y_train_enc.shape)


Encoded shapes:
X_train_enc: (71273, 8)
Y_train_enc: (71273, 7)


In [29]:
lat0 = np.mean(X_train[:, 0])
lon0 = np.mean(X_train[:, 1])

R = 6371000
lat0_rad = np.deg2rad(lat0)

def latlon_to_local(lat, lon):
    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)

    dlat = lat_rad - np.deg2rad(lat0)
    dlon = lon_rad - np.deg2rad(lon0)

    north = dlat * R
    east = dlon * R * np.cos(lat0_rad)

    return north, east


def replace_latlon(X):
    north, east = latlon_to_local(X[:, 0], X[:, 1])
    X_new = X.copy()
    X_new[:, 0] = north
    X_new[:, 1] = east
    return X_new


X_train_enc = replace_latlon(X_train_enc)
X_test_enc  = replace_latlon(X_test_enc)

Y_train_enc = replace_latlon(Y_train_enc)
Y_test_enc  = replace_latlon(Y_test_enc)

print("Projection complete.")


Projection complete.


In [30]:
print("NaNs in X_train_enc:", np.isnan(X_train_enc).sum())
print("NaNs in Y_train_enc:", np.isnan(Y_train_enc).sum())


NaNs in X_train_enc: 0
NaNs in Y_train_enc: 0


In [31]:
X_mean = X_train_enc.mean(axis=0)
X_std  = X_train_enc.std(axis=0)
Y_mean = Y_train_enc.mean(axis=0)
Y_std  = Y_train_enc.std(axis=0)

X_std[X_std == 0] = 1.0
Y_std[Y_std == 0] = 1.0

X_train_std = (X_train_enc - X_mean) / X_std
X_test_std  = (X_test_enc  - X_mean) / X_std

Y_train_std = (Y_train_enc - Y_mean) / Y_std
Y_test_std  = (Y_test_enc  - Y_mean) / Y_std

print("Standardization complete.")


Standardization complete.


In [32]:
print("Train mean approx 0:", np.mean(X_train_std, axis=0))
print("Train std approx 1:", np.std(X_train_std, axis=0))


Train mean approx 0: [-6.2649562e-08  6.2999135e-08  1.3665202e-05 -3.4312420e-06
 -9.5934604e-07 -1.1801867e-06  2.1725384e-06 -1.8212312e-06]
Train std approx 1: [0.9999959  1.0000002  1.0000057  0.9998953  0.99999434 0.99997085
 0.9999799  1.0000257 ]


In [33]:
X_state_train = X_train_enc[:, :7]
X_state_test  = X_test_enc[:, :7]

Y_train_res = Y_train_enc - X_state_train
Y_test_res  = Y_test_enc  - X_state_test


In [34]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))

class StrongResidualMLP(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=256, output_dim=7, num_blocks=3):
        super().__init__()
        
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.blocks = nn.Sequential(
            *[ResidualBlock(hidden_dim) for _ in range(num_blocks)]
        )
        
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        return self.output_layer(x)


In [35]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train_std, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train_std, dtype=torch.float32)

X_test_t  = torch.tensor(X_test_std, dtype=torch.float32)
Y_test_t  = torch.tensor(Y_test_std, dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, Y_train_t)
test_dataset  = TensorDataset(X_test_t, Y_test_t)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1024, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))


Train batches: 70
Test batches: 31


/home/jacobhardy/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA driver initialization failed, you might not have a CUDA gpu. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [36]:
model = StrongResidualMLP(
    input_dim=8,
    hidden_dim=256,
    output_dim=7,
    num_blocks=3
)

print(model)

model = model.to(device)


StrongResidualMLP(
  (input_layer): Sequential(
    (0): Linear(in_features=8, out_features=256, bias=True)
    (1): ReLU()
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (relu): ReLU()
    )
    (1): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (relu): ReLU()
    )
    (2): ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): ReLU()
        (2): Linear(in_features=256, out_features=256, bias=True)
      )
      (relu): ReLU()
    )
  )
  (output_layer): Linear(in_features=256, out_features=7, bias=True)
)


In [37]:
import torch.optim as optim

criterion = torch.nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [38]:
EPOCHS = 150

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * xb.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    # Validation/Test
    model.eval()
    test_loss = 0.0
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            test_loss += loss.item() * xb.size(0)
    
    test_loss /= len(test_loader.dataset)
    
    print(f"Epoch {epoch+1:02d} | Train MSE: {train_loss:.6f} | Test MSE: {test_loss:.6f}")


Epoch 01 | Train MSE: 0.278921 | Test MSE: 0.029380
Epoch 02 | Train MSE: 0.017610 | Test MSE: 0.018858
Epoch 03 | Train MSE: 0.014034 | Test MSE: 0.017152
Epoch 04 | Train MSE: 0.012948 | Test MSE: 0.015881
Epoch 05 | Train MSE: 0.012215 | Test MSE: 0.014994
Epoch 06 | Train MSE: 0.011702 | Test MSE: 0.014226
Epoch 07 | Train MSE: 0.011319 | Test MSE: 0.013661
Epoch 08 | Train MSE: 0.011055 | Test MSE: 0.013224
Epoch 09 | Train MSE: 0.010839 | Test MSE: 0.012959
Epoch 10 | Train MSE: 0.010651 | Test MSE: 0.012743
Epoch 11 | Train MSE: 0.010532 | Test MSE: 0.012493
Epoch 12 | Train MSE: 0.010376 | Test MSE: 0.012341
Epoch 13 | Train MSE: 0.010231 | Test MSE: 0.012175
Epoch 14 | Train MSE: 0.010097 | Test MSE: 0.012027
Epoch 15 | Train MSE: 0.010005 | Test MSE: 0.011944
Epoch 16 | Train MSE: 0.009917 | Test MSE: 0.011916
Epoch 17 | Train MSE: 0.009853 | Test MSE: 0.011833
Epoch 18 | Train MSE: 0.009745 | Test MSE: 0.011711
Epoch 19 | Train MSE: 0.009632 | Test MSE: 0.011658
Epoch 20 | T

In [39]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: False
CUDA device count: 1


In [40]:
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb)

        all_preds.append(pred.cpu())
        all_targets.append(yb)

pred_std = torch.cat(all_preds, dim=0).numpy()
target_std = torch.cat(all_targets, dim=0).numpy()


In [41]:
pred_residual = pred_std * Y_std + Y_mean
true_residual = target_std * Y_std + Y_mean


In [42]:
X_test_real = X_test_std * X_std + X_mean


In [43]:
pred_next = X_test_real[:, :7] + pred_residual
true_next = X_test_real[:, :7] + true_residual


In [44]:
import numpy as np

R = 6371000  

lat = np.deg2rad(test_pairs_df["lat"].values)
lon = np.deg2rad(test_pairs_df["lon"].values)
lat_next_true = np.deg2rad(test_pairs_df["lat_next"].values)
lon_next_true = np.deg2rad(test_pairs_df["lon_next"].values)

sog = test_pairs_df["sog"].values  # knots
cog = np.deg2rad(test_pairs_df["cog"].values)  # radians
dt_min = test_pairs_df["dt_min"].values

v = sog * 0.514444

dt_sec = dt_min * 60.0

d = v * dt_sec

lat_dr = lat + (d / R) * np.cos(cog)
lon_dr = lon + (d / (R * np.cos(lat))) * np.sin(cog)


In [45]:
lat0 = np.mean(lat)

def to_local_meters(lat_rad, lon_rad):
    x = R * (lon_rad - lon_rad.mean()) * np.cos(lat0)
    y = R * (lat_rad - lat_rad.mean())
    return x, y

x_true, y_true = to_local_meters(lat_next_true, lon_next_true)

x_dr, y_dr = to_local_meters(lat_dr, lon_dr)


In [46]:
j=len(pred_next)

In [47]:
lst = []
for i in range(j):
    lst.append(pred_next[i][0] - true_next[i][0])

total_abs_error = sum(abs(val) for val in lst)
print(total_abs_error/j)

755.69434


In [48]:
lst = []
for i in range(j):
    lst.append(x_dr[i] - x_true[i])

total_abs_error = sum(abs(val) for val in lst)
print(total_abs_error/j)

764.696400110908


In [49]:
pos_error_nn = np.sqrt(
    (pred_next[:,0] - true_next[:,0])**2 +
    (pred_next[:,1] - true_next[:,1])**2
)

print("NN Position RMSE (m):", np.sqrt(np.mean(pos_error_nn**2)))
print("NN Position MAE (m):", np.mean(pos_error_nn))


NN Position RMSE (m): 2996.3547
NN Position MAE (m): 1327.1827


In [50]:
dr_error = np.sqrt((x_dr - x_true)**2 + (y_dr - y_true)**2)

print("DR Position RMSE (m):", np.sqrt(np.mean(dr_error**2)))
print("DR Position MAE (m):", np.mean(dr_error))


DR Position RMSE (m): 2951.6745970305706
DR Position MAE (m): 1030.8462380023896
